<a href="https://colab.research.google.com/github/dongdaran/CSE2035_project/blob/master/ExtractAsset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# ==============================================
# Video sample 5개 hugging face에서 load 후 저장
# ==============================================
import zipfile
import os

# 현재 디렉토리 기준 경로 저장 path 설정
current_dir = os.getcwd()
output_dir = os.path.join(current_dir, "drive/MyDrive/study")
os.makedirs(output_dir, exist_ok=True)

# zip path 설정
zip_path = os.path.join(current_dir, "drive/MyDrive/study/video_file.zip")

# zip 에서 5개 추출
with zipfile.ZipFile(zip_path, 'r') as zip_ref:

    # zip 내부 파일 목록
    file_list = zip_ref.namelist()

    # video 파일만 필터
    video_files = [f for f in file_list if f.endswith(".mp4")]

    # 처음 5개 선택
    selected_files = video_files[:5]

    # extract
    for file in selected_files:
        zip_ref.extract(file, output_dir)

print("Extracted:", selected_files)



Extracted: ['video_file/test/USER00000001/VIDEO00007155.mp4', 'video_file/test/USER00000002/VIDEO00005842.mp4', 'video_file/test/USER00000002/VIDEO00006210.mp4', 'video_file/test/USER00000004/VIDEO00005110.mp4', 'video_file/test/USER00000006/VIDEO00005516.mp4']


# ✈ photoextractor demo(for 1 sample)

In [ ]:
!pip install colorgram.py ultralytics qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 67.4 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations

import os
import re
import cv2
import json
import time
import glob
import queue
import torch
import colorgram
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter
from PIL import Image
from ultralytics import YOLO
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info


# =========================================================
# 0) Constants
# =========================================================
SHOT_SIZE = ["Extreme Wide", "Wide", "Medium Wide", "Medium", "Medium Close Up", "Close Up", "Extreme Close Up"]
SHOT_FRAMING = ["Establishing Shot", "Over the Shoulder", "Single", "2 Shot", "3 Shot", "Group Shot", "Insert"]
CAMERA_ANGLE = ["Aerial", "Overhead", "High Angle", "Low Angle", "Dutch Angle", "Ultra Wide / Fisheye"]
LENS_SIZE = ["Wide", "Medium", "Long Lens"]
LIGHTING_TYPE = [
    "Daylight", "Sunny", "Overcast", "Moonlight", "Artificial Light", "Practical Light",
    "Fluorescent", "Firelight", "Mixed Light", "HMI", "LED", "Tungsten"
]
LIGHTING_CONDITION = [
    "Soft Light", "Hard Light", "High Contrast", "Low Contrast", "Silhouette",
    "Top Light", "Underlight", "Side Light", "Backlight", "Edge Light"
]
COMPOSITION = ["Center", "Left Heavy", "Right Heavy", "Balanced", "Symmetrical", "Short Side"]
CAMERA_MOVEMENT = [
    "Pan Left", "Pan Right", "Tilt Up", "Tilt Down", "Camera Roll",
    "Move Left", "Move Right", "Tracking", "Trucking Left", "Trucking Right",
    "Boom Up", "Boom Down", "Push In", "Pull Out", "Zoom In", "Zoom Out",
    "Dolly Zoom", "Rack Focus", "Arc", "Static Shot"
]

COLOR_TONE = ["warm", "cool", "mixed"]
SATURATION_TONE = ["saturated", "desaturated", "mixed"]

QUESTION = (
    "You are a strict JSON generator.\n"
    "Return ONLY a single valid JSON object. No markdown, no prose, no code fences.\n"
    "Do not output anything before '{' or after '}'.\n"
    "All values MUST be chosen from the provided choices. Do not invent new labels.\n"
    "If unsure, choose the closest option from the choices.\n"
    "\n"
    "SINGLE-CHOICE FIELDS (choose exactly ONE value each):\n"
    f"- shot_size choices: {SHOT_SIZE}\n"
    f"- shot_framing choices: {SHOT_FRAMING}\n"
    f"- camera_angle choices: {CAMERA_ANGLE}\n"
    f"- lens_size choices: {LENS_SIZE}\n"
    f"- lighting_type choices: {LIGHTING_TYPE}\n"
    f"- lighting_condition choices: {LIGHTING_CONDITION}\n"
    f"- composition choices: {COMPOSITION}\n"
    "\n"
    "MULTI-CHOICE FIELD (ONLY this field can contain multiple items):\n"
    f"- camera_movement atomic choices: {CAMERA_MOVEMENT}\n"
    "  Rules for camera_movement:\n"
    "  1) Output camera_movement as a single STRING containing one or more atomic movement labels.\n"
    "  2) Separator MUST be a comma ',' ONLY. No other separators. No spaces. Example: \"Boom Up,Pan Left\".\n"
    "  3) Time order MUST follow frames: earliest frame -> latest frame.\n"
    "     Example: if frame1 indicates A and frame2 indicates B, output \"A,B\".\n"
    "  4) If a frame indicates a combined label like \"Boom Up and Pan Left\", you MUST split it into atomic labels:\n"
    "     \"Boom Up,Pan Left\" (and keep time order).\n"
    "  5) If only one movement, output just \"A\".\n"
    "\n"
    "REQUIRED KEYS (exactly these keys, all required):\n"
    "shot_size, shot_framing, camera_angle, lens_size, lighting_type, lighting_condition, composition, camera_movement\n"
    "\n"
    "OUTPUT TEMPLATE (fill values):\n"
    "{\"shot_size\":\"...\",\"shot_framing\":\"...\",\"camera_angle\":\"...\",\"lens_size\":\"...\","
    "\"lighting_type\":\"...\",\"lighting_condition\":\"...\",\"composition\":\"...\",\"camera_movement\":\"A,B\"}\n"
)


# =========================================================
# 1) Config
# =========================================================
@dataclass
class PipelineConfig:
    # ShotVL
    shotvl_model_id: str = "Vchitect/ShotVL-3B"
    shotvl_attn_implementation: str = "sdpa"
    shotvl_dtype: torch.dtype = torch.bfloat16
    shotvl_max_new_tokens: int = 64
    shotvl_video_fps: int = 1
    shotvl_max_pixels: int = 360 * 640

    # YOLO
    yolo_model: str = "yolo26n.pt"
    yolo_conf: float = 0.65
    yolo_device: str = "cpu"

    # Color / Sampling
    palette_k: int = 8
    sample_step_sec: float = 1.0

    # Device
    shotvl_device: str = "cuda" if torch.cuda.is_available() else "cpu"
    yolo_device: str = "cpu"


# =========================================================
# 3) ShotVL Extractor
# =========================================================
class ShotVLExtractor:
    def __init__(self, config: PipelineConfig, question: str = QUESTION):
        self.cfg = config
        self.question = question
        self.device = torch.device(config.shotvl_device)

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.cfg.shotvl_model_id,
            attn_implementation=self.cfg.shotvl_attn_implementation,
            torch_dtype=self.cfg.shotvl_dtype,
        ).to(self.device).eval()

        self.processor = AutoProcessor.from_pretrained(
            self.cfg.shotvl_model_id,
            use_fast=True
        )

    def _build_messages(
        self,
        video_path: str,
        max_pixels: Optional[int] = None,
        fps: Optional[int] = None
    ) -> List[Dict[str, Any]]:
        max_pixels = max_pixels if max_pixels is not None else self.cfg.shotvl_max_pixels
        fps = fps if fps is not None else self.cfg.shotvl_video_fps

        msgs = [
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": video_path, "max_pixels": max_pixels, "fps": fps},
                    {"type": "text", "text": self.question},
                ],
            }
        ]
        return msgs

    @staticmethod
    def safe_choice(v: Optional[str], allowed: List[str]) -> Optional[str]:
        result = ""
        if v is None:
            return None

        if "," in v:
            elements = v.split(",")
            for e in elements:
                e = e.strip()
                if e in allowed:
                    result += e + ","
            if result:
                return result[:-1]

        if v in allowed:
            return v

        vv = str(v).strip().lower()
        for a in allowed:
            if a.lower() == vv:
                return a
        return None

    @classmethod
    def pick(cls, raw: Dict[str, Any], k: str, allowed: List[str]) -> Optional[str]:
        v = raw.get(k)
        return cls.safe_choice(v, allowed) if isinstance(v, str) else None

    def generation(self, msgs: List[Dict[str, Any]]) -> Dict[str, Any]:
        text = self.processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(msgs)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.device)

        t0 = time.time()
        with torch.inference_mode():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.cfg.shotvl_max_new_tokens
            )
        elapsed = time.time() - t0

        input_ids_cpu = inputs["input_ids"].detach().cpu()
        out_ids_cpu = out_ids.detach().cpu()
        trimmed = [o[len(i):] for i, o in zip(input_ids_cpu, out_ids_cpu)]
        raw_text = self.processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        print(1)
        print(raw_text)
        raw_text = re.sub(r"^\s*```(?:json)?\s*", "", raw_text)
        raw_text = re.sub(r"\s*```\s*$", "", raw_text)
        print(2)
        print(raw_text)
        try:
            raw_json = json.loads(raw_text)
        except Exception:
            raw_json = {"_raw_text": raw_text}

        result_data = {
            "shot_size": self.pick(raw_json, "shot_size", SHOT_SIZE),
            "shot_framing": self.pick(raw_json, "shot_framing", SHOT_FRAMING),
            "camera_angle": self.pick(raw_json, "camera_angle", CAMERA_ANGLE),
            "lens_size": self.pick(raw_json, "lens_size", LENS_SIZE),
            "lighting_type": self.pick(raw_json, "lighting_type", LIGHTING_TYPE),
            "lighting_condition": self.pick(raw_json, "lighting_condition", LIGHTING_CONDITION),
            "composition": self.pick(raw_json, "composition", COMPOSITION),
            "camera_movement": self.pick(raw_json, "camera_movement", CAMERA_MOVEMENT),
        }

        del inputs, out_ids, image_inputs, video_inputs, input_ids_cpu, out_ids_cpu
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return result_data

    def extract(self, video_path: str) -> Dict[str, Any]:
        # inference
        msgs = self._build_messages(video_path)
        result = self.generation(msgs)

        # save
        part = video_path.split('/')
        user_id = part[-2]
        video_name = part[-1]

        save_dir = os.path.join(os.getcwd(),"drive/MyDrive/study/shotvl_results/", user_id)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f"{video_name}.json")

        with open(save_path, "w", encoding="utf-8") as f:
          json.dump(result, f, ensure_ascii=False, indent=2)
        return result

In [ ]:
# =========================================================
# 4) YOLO Extractor
# =========================================================
class YOLOExtractor:
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.model = YOLO(self.cfg.yolo_model).to(self.cfg.yolo_device)
        self.img_size = (640, 360)

    def _sample_frames_every(
        self,
        video_path: str,
        step_sec: Optional[float] = None
    ) -> List[np.ndarray]:
        step_sec = step_sec if step_sec is not None else self.cfg.sample_step_sec

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS)
        duration_sec = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps if fps and fps > 0 else None

        frames: List[np.ndarray] = []
        t = 0.0

        while True:
            cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
            ok, frame = cap.read()
            if not ok:
                break

            frames.append(frame)
            t += step_sec

            if duration_sec is not None and t > duration_sec + 0.01:
                break

        cap.release()
        return frames

    def detect_objects_on_sampled_frames(
        self,
        frames: List[np.ndarray],
        conf: Optional[float] = None
    ) -> List[List[str]]:
        conf = conf if conf is not None else self.cfg.yolo_conf

        out = []
        results = self.model(frames, verbose=False, conf=conf)

        for result in results:
            names = result.names
            if result.boxes is None or len(result.boxes) == 0:
                objs = []
            else:
                objs = [names[int(b.cls)] for b in result.boxes]
            out.append(objs)

        return out

    def yolo_inference(
        self,
        video_path: str,
        frames: List[np.ndarray],
        fps_mode: int = 1
    ) -> Dict[str, Any]:
        t0 = time.time()
        objs = self.detect_objects_on_sampled_frames(frames=frames, conf=self.cfg.yolo_conf)

        c = Counter()
        for obj_list in objs:
            c.update(obj_list)

        total = sum(c.values()) if sum(c.values()) > 0 else 1

        top5 = []
        for obj, cnt in c.most_common(5):
            top5.append({
                "object": obj,
                "share": cnt / total
            })

        return top5
    @staticmethod
    def _rgb_to_hsv(rgb: Tuple[int, int, int]) -> Tuple[float, float, float]:
        r, g, b = [x / 255.0 for x in rgb]
        mx = max(r, g, b)
        mn = min(r, g, b)
        diff = mx - mn

        if diff == 0:
            h = 0.0
        elif mx == r:
            h = (60 * ((g - b) / diff) + 360) % 360
        elif mx == g:
            h = (60 * ((b - r) / diff) + 120) % 360
        else:
            h = (60 * ((r - g) / diff) + 240) % 360

        s = 0.0 if mx == 0 else diff / mx
        v = mx
        return h, s, v

    def _derive_warm_cool(self, palette: List[Dict[str, Any]]) -> str:
      warm_w, cool_w, total_w = 0.0, 0.0, 0.0

      for p in palette:
          w = float(p.get("proportion", 0.0))
          h, s, v = self._rgb_to_hsv(tuple(p["rgb"]))
          total_w += w

          if (h >= 330) or (h <= 70):
              warm_w += w
          elif 70 < h <= 250:
              cool_w += w
          else:
              # 경계에 있는 색상은 양쪽에 절반씩 기여
              warm_w += 0.5 * w
              cool_w += 0.5 * w

      if total_w <= 0: return "mixed"

      # 전체 비중 합으로 나누어 정규화
      warm_ratio = warm_w / total_w
      cool_ratio = cool_w / total_w

      if warm_ratio >= 0.60: return "warm"
      if cool_ratio >= 0.60: return "cool"
      return "mixed"

    def _derive_saturation(self, palette: List[Dict[str, Any]]) -> str:
      s_weighted_sum, w_sum = 0.0, 0.0

      for p in palette:
          w = float(p.get("proportion", 0.0))
          h, s, v = self._rgb_to_hsv(tuple(p["rgb"]))

          s_weighted_sum += s * w
          w_sum += w

      if w_sum <= 0: return "mixed"

      s_avg = s_weighted_sum / w_sum

      if s_avg >= 0.55: return "saturated"
      if s_avg <= 0.30: return "desaturated"
      return "mixed"

    def extra_info_inference(self, frames):
      total_palette = []

      for frame_bgr in frames:
          rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
          pil_img = Image.fromarray(rgb).resize(self.img_size, Image.BILINEAR)

          # 색상 추출 (최대 k개)
          colors = colorgram.extract(pil_img, self.cfg.palette_k)
          total_p = sum([c.proportion for c in colors]) or 1.0

          # 모든 프레임의 데이터를 하나의 리스트에 확장(extend)
          total_palette.extend([
              {
                  "rgb": [int(c.rgb.r), int(c.rgb.g), int(c.rgb.b)],
                  "proportion": float(c.proportion) / total_p
              } for c in colors
          ])

      # 통합 팔레트를 기반으로 최종 결과 도출
      warm_cool_result = self._derive_warm_cool(total_palette)
      saturation_result = self._derive_saturation(total_palette)

      return warm_cool_result, saturation_result

    def extract(self, video_path: str) -> Dict[str, Any]:
        frames = self._sample_frames_every(video_path, step_sec=self.cfg.sample_step_sec)

        fps_mode = int(round(1 / self.cfg.sample_step_sec)) if self.cfg.sample_step_sec > 0 else 1

        yolo_result = self.yolo_inference(
            video_path=video_path,
            frames=frames,
            fps_mode=fps_mode
        )
        warm_result, saturation_result = self.extra_info_inference(frames)
        result = {
            "top5" : yolo_result,
            "warm_cool": warm_result,
            "saturation": saturation_result
        }

        # save
        part = video_path.split('/')
        user_id = part[-2]
        video_name = part[-1]

        save_dir = os.path.join(os.getcwd(),"drive/MyDrive/study/extra_results", user_id)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f"{video_name}.json")

        with open(save_path, "w", encoding="utf-8") as f:
          json.dump(result, f, ensure_ascii=False, indent=2)

        return result

In [ ]:
cfg = PipelineConfig()

video_path = "/content/drive/MyDrive/study/video_file/test/USER00000006/VIDEO00005516.mp4"

shotvl_extractor = ShotVLExtractor(cfg)
yolo_extractor = YOLOExtractor(cfg)

shotvl_result = shotvl_extractor.extract(video_path)
extra_result = yolo_extractor.extract(video_path)

final_result = {
    "video_path": video_path,
    "shotvl": shotvl_result,
    "extra": extra_result
}

print(json.dumps(final_result, ensure_ascii=False, indent=2))

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

1
{"shot_size":"Medium Wide","shot_framing":"Group Shot","camera_angle":"High Angle","lens_size":"Medium","lighting_type":"Artificial Light","lighting_condition":"Soft Light","composition":"Balanced","camera_movement":"Pan Left"}
2
{"shot_size":"Medium Wide","shot_framing":"Group Shot","camera_angle":"High Angle","lens_size":"Medium","lighting_type":"Artificial Light","lighting_condition":"Soft Light","composition":"Balanced","camera_movement":"Pan Left"}
{
  "video_path": "/content/drive/MyDrive/study/video_file/test/USER00000006/VIDEO00005516.mp4",
  "shotvl": {
    "shot_size": "Medium Wide",
    "shot_framing": "Group Shot",
    "camera_angle": "High Angle",
    "lens_size": "Medium",
    "lighting_type": "Artificial Light",
    "lighting_condition": "Soft Light",
    "composition": "Balanced",
    "camera_movement": "Pan Left"
  },
  "extra": {
    "top5": [
      {
        "object": "person",
        "share": 0.907608695652174
      },
      {
        "object": "potted plant",
  

In [ ]:
def clear_gpu():
    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
clear_gpu()

# ⁉ shotvl can receive frame input?
# 🏃shortvl inferenc 속도 측정


- yolo와 shotvl input을 일치시키기 위한 시도 -> 실패!. 오류가 나요~~

- yolo와 shotvl을 맞추기 위해 내가 꼭 해야할까....? 그냥 따로해



In [ ]:
!pip install qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 68.7 MB/s eta 0:00:00


In [ ]:
from typing import Any, Dict, List, Optional, Tuple
SHOT_SIZE = ["Extreme Wide", "Wide", "Medium Wide", "Medium", "Medium Close Up", "Close Up", "Extreme Close Up"]
SHOT_FRAMING = ["Establishing Shot", "Over the Shoulder", "Single", "2 Shot", "3 Shot", "Group Shot", "Insert"]
CAMERA_ANGLE = ["Aerial", "Overhead", "High Angle", "Low Angle", "Dutch Angle", "Ultra Wide / Fisheye"]
LENS_SIZE = ["Wide", "Medium", "Long Lens"]
LIGHTING_TYPE = [
    "Daylight", "Sunny", "Overcast", "Moonlight", "Artificial Light", "Practical Light",
    "Fluorescent", "Firelight", "Mixed Light", "HMI", "LED", "Tungsten"
]
LIGHTING_CONDITION = [
    "Soft Light", "Hard Light", "High Contrast", "Low Contrast", "Silhouette",
    "Top Light", "Underlight", "Side Light", "Backlight", "Edge Light"
]
COMPOSITION = ["Center", "Left Heavy", "Right Heavy", "Balanced", "Symmetrical", "Short Side"]
CAMERA_MOVEMENT = [
    "Pan Left", "Pan Right", "Tilt Up", "Tilt Down", "Camera Roll",
    "Move Left", "Move Right", "Tracking", "Trucking Left", "Trucking Right",
    "Boom Up", "Boom Down", "Push In", "Pull Out", "Zoom In", "Zoom Out",
    "Dolly Zoom", "Rack Focus", "Arc", "Static Shot"
]


def safe_choice(v: Optional[str], allowed: List[str]) -> Optional[str]:
    result = ""
    if v is None:
        return None

    if "," in v:
      elements = v.split(",")
      for e in elements:
        e = e.strip()
        ee = str(e).strip()
        if e in allowed:
          result += e + ","
        elif ee in allowed:
          result += ee + ","
      if result:
        return result[:-1]


    if v in allowed:
        return v
    vv = str(v).strip().lower()
    for a in allowed:
        if a.lower() == vv:
            return a
    return None

def pick(k: str, allowed: List[str]) -> Optional[str]:
    v = raw.get(k)
    return safe_choice(v, allowed) if isinstance(v, str) else None

In [ ]:
import cv2
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import time
import glob
import re
import os
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_map = None
dtype = torch.bfloat16
# load all file path
video_paths = glob.glob(os.path.join(
    os.getcwd(),
    "drive/MyDrive/study/video_file/test",
    "**",
    "*.mp4"
), recursive=True)

#frames = sample_frames_every(video_path, step_sec=1)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
  "Vchitect/ShotVL-3B",
  device_map=device_map,
  attn_implementation= "sdpa",
  torch_dtype=dtype,
).to(device).eval()


processor = AutoProcessor.from_pretrained(
  "Vchitect/ShotVL-3B", use_fast=True, dtype=torch.bfloat16
)

question = (
    "You are a strict JSON generator.\n"
    "Return ONLY a single valid JSON object. No markdown, no prose, no code fences.\n"
    "Do not output anything before '{' or after '}'.\n"
    "All values MUST be chosen from the provided choices. Do not invent new labels.\n"
    "If unsure, choose the closest option from the choices.\n"
    "\n"
    "SINGLE-CHOICE FIELDS (choose exactly ONE value each):\n"
    f"- shot_size choices: {SHOT_SIZE}\n"
    f"- shot_framing choices: {SHOT_FRAMING}\n"
    f"- camera_angle choices: {CAMERA_ANGLE}\n"
    f"- lens_size choices: {LENS_SIZE}\n"
    f"- lighting_type choices: {LIGHTING_TYPE}\n"
    f"- lighting_condition choices: {LIGHTING_CONDITION}\n"
    f"- composition choices: {COMPOSITION}\n"
    "\n"
    "MULTI-CHOICE FIELD (ONLY this field can contain multiple items):\n"
    f"- camera_movement atomic choices: {CAMERA_MOVEMENT}\n"
    "  Rules for camera_movement:\n"
    "  1) Output camera_movement as a single STRING containing one or more atomic movement labels.\n"
    "  2) Separator MUST be a comma ',' ONLY. No other separators. No spaces. Example: \"Boom Up,Pan Left\".\n"
    "  3) Time order MUST follow frames: earliest frame -> latest frame.\n"
    "     Example: if frame1 indicates A and frame2 indicates B, output \"A,B\".\n"
    "  4) If a frame indicates a combined label like \"Boom Up and Pan Left\", you MUST split it into atomic labels:\n"
    "     \"Boom Up,Pan Left\" (and keep time order).\n"
    "  5) If only one movement, output just \"A\".\n"
    "\n"
    "REQUIRED KEYS (exactly these keys, all required):\n"
    "shot_size, shot_framing, camera_angle, lens_size, lighting_type, lighting_condition, composition, camera_movement\n"
    "\n"
    "OUTPUT TEMPLATE (fill values):\n"
    "{\"shot_size\":\"...\",\"shot_framing\":\"...\",\"camera_angle\":\"...\",\"lens_size\":\"...\","
    "\"lighting_type\":\"...\",\"lighting_condition\":\"...\",\"composition\":\"...\",\"camera_movement\":\"A,B\"}\n"
)

for video_path in video_paths:
  t0 = time.time()
  msgs = [
    {
      "role": "user",
      "content": [
        {"type": "video", "video": video_path, "max_pixels": 360 * 640, "fps": 1},
        {"type": "text", "text": question},
      ],
    },
  ]

  text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
  image_inputs, video_inputs = process_vision_info(msgs)
  inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    use_cache =False,
    return_tensors="pt",
  ).to("cpu")

  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.inference_mode():
    out_ids = model.generate(**inputs, max_new_tokens=64)

  input_ids_cpu = inputs["input_ids"].detach().cpu()
  out_ids_cpu = out_ids.detach().cpu()

  trimmed = [o[len(i):] for i, o in zip(input_ids_cpu, out_ids_cpu)]
  raw = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

  raw = raw.strip()
  # ```json ... ``` 또는 ``` ... ``` 제거
  raw = re.sub(r"^\s*```(?:json)?\s*", "", raw)
  raw = re.sub(r"\s*```\s*$", "", raw)

  print(raw)

  try:
      raw = json.loads(raw)
  except Exception:
      raw = {"_raw_text": raw}
  print(type(raw))

  print("="*40)
  print(f"🎬 Camera Extraction Result{"/".join(video_path.split('/')[-2:])}")
  print("="*40)

  result_data = {
        "shot_size": pick('shot_size', SHOT_SIZE),
        "shot_framing": pick('shot_framing', SHOT_FRAMING),
        "camera_angle": pick('camera_angle', CAMERA_ANGLE),
        "lens_size": pick('lens_size', LENS_SIZE),
        "lighting_type": pick('lighting_type', LIGHTING_TYPE),
        "lighting_condition": pick('lighting_condition', LIGHTING_CONDITION),
        "composition": pick('composition', COMPOSITION),
        "camera_movement": pick('camera_movement',CAMERA_MOVEMENT)
    }

  for key, val in result_data.items():
      print(f"{key.replace('_', ' ').title():<18}: {val}")

  print(f"⏱️ Time Taken      : {time.time()-t0:.2f}s")
  print("="*40 + "\n")

  # [메모리 관리] 현재 루프의 대형 객체들 삭제 및 캐시 비우기
  del inputs, out_ids, image_inputs, video_inputs, input_ids_cpu, out_ids_cpu
  if torch.cuda.is_available():
      torch.cuda.empty_cache()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

qwen-vl-utils using torchcodec to read video.
Keyword argument `use_cache` is not a valid argument for this processor and will be ignored.


{"shot_size":"Medium","shot_framing":"Single","camera_angle":"Dutch Angle","lens_size":"Wide","lighting_type":"Artificial Light","lighting_condition":"High Contrast","composition":"Center","camera_movement":"Pan Left"}
<class 'dict'>
🎬 Camera Extraction ResultUSER00000001/VIDEO00007155.mp4
Shot Size         : Medium
Shot Framing      : Single
Camera Angle      : Dutch Angle
Lens Size         : Wide
Lighting Type     : Artificial Light
Lighting Condition: High Contrast
Composition       : Center
Camera Movement   : Pan Left
⏱️ Time Taken      : 4.73s

{"shot_size":"Close Up","shot_framing":"Single","camera_angle":"High Angle","lens_size":"Long Lens","lighting_type":"Daylight","lighting_condition":"Hard Light","composition":"Center","camera_movement":"Dolly Zoom"}
<class 'dict'>
🎬 Camera Extraction ResultUSER00000002/VIDEO00005842.mp4
Shot Size         : Close Up
Shot Framing      : Single
Camera Angle      : High Angle
Lens Size         : Long Lens
Lighting Type     : Daylight
Lighting 

# ✅ object detector demo

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.3 MB/s eta 0:00:00


In [ ]:
import cv2
import os
from typing import List, Tuple

def sample_frames_every(video_path: str, step_sec: float = 0.5):
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"Cannot open video: {video_path}"

    fps = cap.get(cv2.CAP_PROP_FPS)  # 해당 video의 fps 구하기 -> e.g.29.97
    duration_sec = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps if fps > 0 else None # 동영상 길이 구하기

    frames: List[Tuple[float, any]] = []  # (t_sec, frame BGR)

    t = 0.0
    while True:
        # 원하는 시점으로 점프 (밀리초 단위)
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        # frame 저장
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
        # 다음 frame으로 이동
        t += step_sec

        # 안전장치: duration 알면 넘기면 종료
        if duration_sec is not None and t > duration_sec + 0.01:
            break

    cap.release()
    return frames

In [ ]:
from ultralytics import YOLO

def detect_objects_on_sampled_frames(model, frames, conf: float = 0.65):
    # model = YOLO("yolov8n.pt")

    out = []
    results = model(frames, verbose=False, conf=conf)
    device = next(model.model.parameters()).device
    for result in results:
        names = result.names
        if result.boxes is None and len(result.boxes) > 0:
          objs = []
        else:
          cls_names = [names[int(b.cls)] for b in result.boxes]
          objs = cls_names
        out.append(objs)
    return out

In [ ]:
from collections import Counter
import glob
import pandas as pd
import time

t0 = time.time()

model = YOLO("yolo26n.pt").to("cpu")

results = {}
meta = {}

# load all file path
video_paths = glob.glob(os.path.join(
    os.getcwd(),
    "drive/MyDrive/study/video_file/test",
    "**",
    "*.mp4"
), recursive=True)

# fps_modes : 1fps, 2fps
# fps_modes = {1, 2}
fps_modes = {1}

for vp in video_paths:
  for fm in fps_modes:
    tt1 = time.time()
    frames = sample_frames_every(vp, step_sec=1/fm)
    objs = detect_objects_on_sampled_frames(model=model, frames =frames)

    c = Counter()
    for obj in objs:
      c.update(obj)

    results[(vp, fm)] = c
    meta[(vp, fm)] = { "sampled_frames": len(objs)}


  # 영상별 Top5 테이블 생성
  rows = []
  for (vp, fm), counter in results.items():
      total = sum(counter.values()) if sum(counter.values()) > 0 else 1
      for obj, cnt in counter.most_common(5):
          rows.append({
              "video": vp,
              "fps_mode": fm,
              "object": obj,
              "count": cnt,
              "share": cnt / total,
              "sampled_frames": meta[(vp, fm)]["sampled_frames"],
          })
  print(f"⭐ 걸린 시간({vp}) : {time.time()-tt1}")

  df_topk = pd.DataFrame(rows).sort_values(["video", "fps_mode", "count"], ascending=[True, True, False])

print("\n=== [영상별 Top-5 Object 분포] ===")
print(df_topk.to_string(index=False))

print("\n=== [메타 정보] (샘플링 프레임 수 등) ===")
for k, v in meta.items():
    vp, fm = k
    print(f"- {os.path.basename(vp)} | {fm}fps: {v}")

print("\n=== [걸린 시간]===")
print(time.time() - t0)

⭐ 걸린 시간(/content/drive/MyDrive/study/video_file/test/USER00000001/VIDEO00007155.mp4) : 1.555492639541626
⭐ 걸린 시간(/content/drive/MyDrive/study/video_file/test/USER00000002/VIDEO00005842.mp4) : 2.716235876083374
⭐ 걸린 시간(/content/drive/MyDrive/study/video_file/test/USER00000002/VIDEO00006210.mp4) : 3.3117361068725586
⭐ 걸린 시간(/content/drive/MyDrive/study/video_file/test/USER00000004/VIDEO00005110.mp4) : 1.0871610641479492
⭐ 걸린 시간(/content/drive/MyDrive/study/video_file/test/USER00000006/VIDEO00005516.mp4) : 3.5759358406066895

=== [영상별 Top-5 Object 분포] ===
                                                                      video  fps_mode       object  count    share  sampled_frames
/content/drive/MyDrive/study/video_file/test/USER00000001/VIDEO00007155.mp4         1       person     29 1.000000              16
/content/drive/MyDrive/study/video_file/test/USER00000002/VIDEO00005842.mp4         1       person      6 0.428571              43
/content/drive/MyDrive/study/video_file/test/USE

In [ ]:
df = df_topk.copy()
df['video_name'] = df['video'].apply(lambda x: x.split('/')[-1])

# 3. 비디오와 fps_mode별로 그룹화하여 처리
grouped = df.groupby(['video_name', 'fps_mode'])

print("### 비디오 및 FPS별 객체 분포 비교 ###\n")

# 결과를 저장하거나 출력
for (video, fps), group in grouped:
    # share 기준 내림차순 정렬 후 딕셔너리 변환
    sorted_group = group.sort_values(by='share', ascending=False)
    distribution = dict(zip(sorted_group['object'], sorted_group['share']))

    print(f"Video: {video} | FPS Mode: {fps}")
    print(f"Distribution: {distribution}")
    print("-" * 50)

In [ ]:
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import os

current_dir = os.getcwd()
video_path = os.path.join(current_dir, "drive/MyDrive/study/video_file/test/USER00000006/VIDEO00005516.mp4")


# Load the YOLO26 model
model = YOLO("yolo26n.pt")

# Open the webcam
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
# if cap.set(cv2.CAP_PROP_FPS, 30):
# 	print("FPS set")
# else:
# 	print("FPS setting failed")

cap = cv2.VideoCapture(video_path)

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO26 tracking on the frame, persisting tracks between frames
        results = model.track(frame, persist=True, verbose=False)

        # Visualize the results on the frame
        annotated_frame = results[0].plot()

        print(f"Number of objects: {len(results[0])}")
        #print(results[0].boxes.id) # (tensor) 바운딩 박스의 track ID를 반환합니다 (사용 가능한 경우).
        #print(results[0].names) # (dict) 클래스 인덱스를 클래스 이름에 매핑하는 사전입니다.
        #print(results[0].boxes.cls) # (tensor) bounding box의 클래스 값을 반환합니다.

        # ID와 각 물체의 클래스 출력
        for id, cls in zip(results[0].boxes.id, results[0].boxes.cls):
            print(f"ID: {id.int()} Class: {results[0].names[cls.int().item()]}", end=', ')
        print()

        # Display the annotated frame
        cv2_imshow(annotated_frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

# ✂ scene cutting model demo

In [ ]:
!pip install scenedetect[opencv]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.9/130.9 kB 14.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.


In [ ]:
import os
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector
import time

t0 = time.time()

current_dir = os.getcwd()
video_path = os.path.join(
    current_dir,
    "drive/MyDrive/study/video_file/test/USER00000006/VIDEO00005516.mp4"
)

if not os.path.exists(video_path):
    raise FileNotFoundError(video_path)

# threshold 낮을수록 더 민감(씬이 더 많이 잡힘)
for threshold in range(65, 70):
    # 루프마다 새로 만들어야 detector 누적/재생 위치 문제 없음
    video = open_video(video_path)  # 새 스트림
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=threshold))

    scene_manager.detect_scenes(video, show_progress=False)
    scenes = scene_manager.get_scene_list()

    print(f"threshold: {threshold}")
    print(f"scenes len: {len(scenes)}")

    # 씬 구간 출력(원하면)
    # for i, (start, end) in enumerate(scenes):
    #     print(f"Scene {i}: {start.get_seconds():.2f} -> {end.get_seconds():.2f}")

INFO:pyscenedetect:Detecting scenes...
INFO:pyscenedetect:Detecting scenes...


threshold: 65
scenes len: 12


INFO:pyscenedetect:Detecting scenes...


threshold: 66
scenes len: 9


INFO:pyscenedetect:Detecting scenes...


threshold: 67
scenes len: 5


INFO:pyscenedetect:Detecting scenes...


threshold: 68
scenes len: 5
threshold: 69
scenes len: 5


# ✈ photoextractor final version(applied parallel computing)

In [ ]:
!pip install colorgram.py ultralytics qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 59.8 MB/s eta 0:00:00


In [ ]:
!pip install -U "ray[data,train,tune,serve]"

INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 147.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 41.6 MB/s eta 0:00:00


In [ ]:
import ray

if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus = 12, num_gpus = 1)

2026-03-13 14:08:23,205	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.54.0
Dashboard:,http://127.0.0.1:8265


In [1]:
from __future__ import annotations

import os
import re
import cv2
import json
import time
import glob
import queue
import torch
import colorgram
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter
from PIL import Image
from ultralytics import YOLO
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from vllm import LLM, SamplingParams


# =========================================================
# 0) Constants
# =========================================================
SHOT_SIZE = ["Extreme Wide", "Wide", "Medium Wide", "Medium", "Medium Close Up", "Close Up", "Extreme Close Up"]
SHOT_FRAMING = ["Establishing Shot", "Over the Shoulder", "Single", "2 Shot", "3 Shot", "Group Shot", "Insert"]
CAMERA_ANGLE = ["Aerial", "Overhead", "High Angle", "Low Angle", "Dutch Angle", "Ultra Wide / Fisheye"]
LENS_SIZE = ["Wide", "Medium", "Long Lens"]
LIGHTING_TYPE = [
    "Daylight", "Sunny", "Overcast", "Moonlight", "Artificial Light", "Practical Light",
    "Fluorescent", "Firelight", "Mixed Light", "HMI", "LED", "Tungsten"
]
LIGHTING_CONDITION = [
    "Soft Light", "Hard Light", "High Contrast", "Low Contrast", "Silhouette",
    "Top Light", "Underlight", "Side Light", "Backlight", "Edge Light"
]
COMPOSITION = ["Center", "Left Heavy", "Right Heavy", "Balanced", "Symmetrical", "Short Side"]
CAMERA_MOVEMENT = [
    "Pan Left", "Pan Right", "Tilt Up", "Tilt Down", "Camera Roll",
    "Move Left", "Move Right", "Tracking", "Trucking Left", "Trucking Right",
    "Boom Up", "Boom Down", "Push In", "Pull Out", "Zoom In", "Zoom Out",
    "Dolly Zoom", "Rack Focus", "Arc", "Static Shot"
]

COLOR_TONE = ["warm", "cool", "mixed"]
SATURATION_TONE = ["saturated", "desaturated", "mixed"]

QUESTION = (
    "You are a strict JSON generator.\n"
    "Return ONLY a single valid JSON object. No markdown, no prose, no code fences.\n"
    "Do not output anything before '{' or after '}'.\n"
    "All values MUST be chosen from the provided choices. Do not invent new labels.\n"
    "If unsure, choose the closest option from the choices.\n"
    "\n"
    "SINGLE-CHOICE FIELDS (choose exactly ONE value each):\n"
    f"- shot_size choices: {SHOT_SIZE}\n"
    f"- shot_framing choices: {SHOT_FRAMING}\n"
    f"- camera_angle choices: {CAMERA_ANGLE}\n"
    f"- lens_size choices: {LENS_SIZE}\n"
    f"- lighting_type choices: {LIGHTING_TYPE}\n"
    f"- lighting_condition choices: {LIGHTING_CONDITION}\n"
    f"- composition choices: {COMPOSITION}\n"
    "\n"
    "MULTI-CHOICE FIELD (ONLY this field can contain multiple items):\n"
    f"- camera_movement atomic choices: {CAMERA_MOVEMENT}\n"
    "  Rules for camera_movement:\n"
    "  1) Output camera_movement as a single STRING containing one or more atomic movement labels.\n"
    "  2) Separator MUST be a comma ',' ONLY. No other separators. No spaces. Example: \"Boom Up,Pan Left\".\n"
    "  3) Time order MUST follow frames: earliest frame -> latest frame.\n"
    "     Example: if frame1 indicates A and frame2 indicates B, output \"A,B\".\n"
    "  4) If a frame indicates a combined label like \"Boom Up and Pan Left\", you MUST split it into atomic labels:\n"
    "     \"Boom Up,Pan Left\" (and keep time order).\n"
    "  5) If only one movement, output just \"A\".\n"
    "\n"
    "REQUIRED KEYS (exactly these keys, all required):\n"
    "shot_size, shot_framing, camera_angle, lens_size, lighting_type, lighting_condition, composition, camera_movement\n"
    "\n"
    "OUTPUT TEMPLATE (fill values):\n"
    "{\"shot_size\":\"...\",\"shot_framing\":\"...\",\"camera_angle\":\"...\",\"lens_size\":\"...\","
    "\"lighting_type\":\"...\",\"lighting_condition\":\"...\",\"composition\":\"...\",\"camera_movement\":\"A,B\"}\n"
)


# =========================================================
# 1) Config
# =========================================================
@dataclass
class PipelineConfig:
    # ShotVL
    shotvl_model_id: str = "Vchitect/ShotVL-3B"
    shotvl_attn_implementation: str = "sdpa"
    shotvl_dtype: torch.dtype = torch.bfloat16
    shotvl_max_new_tokens: int = 64
    shotvl_video_fps: int = 1
    shotvl_max_pixels: int = 360 * 640

    # YOLO
    yolo_model: str = "/content/yolo26n.pt"
    yolo_conf: float = 0.65
    yolo_device: str = "cpu"

    # Color / Sampling
    palette_k: int = 8
    sample_step_sec: float = 1.0

    # Device
    shotvl_device: str = "cuda" if torch.cuda.is_available() else "cpu"
    yolo_device: str = "cpu"


# =========================================================
# 3) ShotVL Extractor
# =========================================================
class ShotVLExtractor:
    def __init__(self, config: PipelineConfig, question: str = QUESTION):
        self.cfg = config
        self.question = question
        self.device = torch.device(config.shotvl_device)

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.cfg.shotvl_model_id,
            attn_implementation=self.cfg.shotvl_attn_implementation,
            torch_dtype=self.cfg.shotvl_dtype,
        ).to(self.device).eval()

        self.processor = AutoProcessor.from_pretrained(
            self.cfg.shotvl_model_id,
            use_fast=True
        )

    def _build_messages(
        self,
        video_path: str,
        max_pixels: Optional[int] = None,
        fps: Optional[int] = None
    ) -> List[Dict[str, Any]]:
        max_pixels = max_pixels if max_pixels is not None else self.cfg.shotvl_max_pixels
        fps = fps if fps is not None else self.cfg.shotvl_video_fps

        msgs = [
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": video_path, "max_pixels": max_pixels, "fps": fps},
                    {"type": "text", "text": self.question},
                ],
            }
        ]
        return msgs

    @staticmethod
    def safe_choice(v: Optional[str], allowed: List[str]) -> Optional[str]:
        result = ""
        if v is None:
            return None

        if "," in v:
            elements = v.split(",")
            for e in elements:
                e = e.strip()
                if e in allowed:
                    result += e + ","
            if result:
                return result[:-1]

        if v in allowed:
            return v

        vv = str(v).strip().lower()
        for a in allowed:
            if a.lower() == vv:
                return a
        return None

    @classmethod
    def pick(cls, raw: Dict[str, Any], k: str, allowed: List[str]) -> Optional[str]:
        v = raw.get(k)
        return cls.safe_choice(v, allowed) if isinstance(v, str) else None

    def generation(self, msgs: List[Dict[str, Any]]) -> Dict[str, Any]:
        text = self.processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(msgs)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.device)

        t0 = time.time()
        with torch.inference_mode():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.cfg.shotvl_max_new_tokens
            )
        elapsed = time.time() - t0

        input_ids_cpu = inputs["input_ids"].detach().cpu()
        out_ids_cpu = out_ids.detach().cpu()
        trimmed = [o[len(i):] for i, o in zip(input_ids_cpu, out_ids_cpu)]
        raw_text = self.processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        raw_text = re.sub(r"^\s*```(?:json)?\s*", "", raw_text)
        raw_text = re.sub(r"\s*```\s*$", "", raw_text)
        try:
            raw_json = json.loads(raw_text)
        except Exception:
            raw_json = {"_raw_text": raw_text}

        result_data = {
            "shot_size": self.pick(raw_json, "shot_size", SHOT_SIZE),
            "shot_framing": self.pick(raw_json, "shot_framing", SHOT_FRAMING),
            "camera_angle": self.pick(raw_json, "camera_angle", CAMERA_ANGLE),
            "lens_size": self.pick(raw_json, "lens_size", LENS_SIZE),
            "lighting_type": self.pick(raw_json, "lighting_type", LIGHTING_TYPE),
            "lighting_condition": self.pick(raw_json, "lighting_condition", LIGHTING_CONDITION),
            "composition": self.pick(raw_json, "composition", COMPOSITION),
            "camera_movement": self.pick(raw_json, "camera_movement", CAMERA_MOVEMENT),
        }

        del inputs, out_ids, image_inputs, video_inputs, input_ids_cpu, out_ids_cpu
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return result_data

    def extract(self, video_path: str) -> Dict[str, Any]:
        # inference
        msgs = self._build_messages(video_path)
        result = self.generation(msgs)

        # save
        part = video_path.split('/')
        user_id = part[-2]
        video_name = part[-1]

        save_dir = os.path.join(os.getcwd(),"drive/MyDrive/study/shotvl_results/", user_id)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f"{video_name}.json")

        with open(save_path, "w", encoding="utf-8") as f:
          json.dump(result, f, ensure_ascii=False, indent=2)
        return result

ModuleNotFoundError: No module named 'colorgram'

In [ ]:
# =========================================================
# 4) YOLO Extractor
# =========================================================
class YOLOExtractor:
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.model = YOLO(self.cfg.yolo_model).to(self.cfg.yolo_device)
        self.img_size = (640, 360)

    def _sample_frames_every(
        self,
        video_path: str,
        step_sec: Optional[float] = None
    ) -> List[np.ndarray]:
        step_sec = step_sec if step_sec is not None else self.cfg.sample_step_sec

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS)
        duration_sec = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps if fps and fps > 0 else None

        frames: List[np.ndarray] = []
        t = 0.0

        while True:
            cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
            ok, frame = cap.read()
            if not ok:
                break

            frames.append(frame)
            t += step_sec

            if duration_sec is not None and t > duration_sec + 0.01:
                break

        cap.release()
        return frames

    def detect_objects_on_sampled_frames(
        self,
        frames: List[np.ndarray],
        conf: Optional[float] = None
    ) -> List[List[str]]:
        conf = conf if conf is not None else self.cfg.yolo_conf

        out = []
        results = self.model(frames, verbose=False, conf=conf)

        for result in results:
            names = result.names
            if result.boxes is None or len(result.boxes) == 0:
                objs = []
            else:
                objs = [names[int(b.cls)] for b in result.boxes]
            out.append(objs)

        return out

    def yolo_inference(
        self,
        video_path: str,
        frames: List[np.ndarray],
        fps_mode: int = 1
    ) -> Dict[str, Any]:
        t0 = time.time()
        objs = self.detect_objects_on_sampled_frames(frames=frames, conf=self.cfg.yolo_conf)

        c = Counter()
        for obj_list in objs:
            c.update(obj_list)

        total = sum(c.values()) if sum(c.values()) > 0 else 1

        top5 = []
        for obj, cnt in c.most_common(5):
            top5.append({
                "object": obj,
                "share": cnt / total
            })

        return top5
    @staticmethod
    def _rgb_to_hsv(rgb: Tuple[int, int, int]) -> Tuple[float, float, float]:
        r, g, b = [x / 255.0 for x in rgb]
        mx = max(r, g, b)
        mn = min(r, g, b)
        diff = mx - mn

        if diff == 0:
            h = 0.0
        elif mx == r:
            h = (60 * ((g - b) / diff) + 360) % 360
        elif mx == g:
            h = (60 * ((b - r) / diff) + 120) % 360
        else:
            h = (60 * ((r - g) / diff) + 240) % 360

        s = 0.0 if mx == 0 else diff / mx
        v = mx
        return h, s, v

    def _derive_warm_cool(self, palette: List[Dict[str, Any]]) -> str:
      warm_w, cool_w, total_w = 0.0, 0.0, 0.0

      for p in palette:
          w = float(p.get("proportion", 0.0))
          h, s, v = self._rgb_to_hsv(tuple(p["rgb"]))
          total_w += w

          if (h >= 330) or (h <= 70):
              warm_w += w
          elif 70 < h <= 250:
              cool_w += w
          else:
              # 경계에 있는 색상은 양쪽에 절반씩 기여
              warm_w += 0.5 * w
              cool_w += 0.5 * w

      if total_w <= 0: return "mixed"

      # 전체 비중 합으로 나누어 정규화
      warm_ratio = warm_w / total_w
      cool_ratio = cool_w / total_w

      if warm_ratio >= 0.60: return "warm"
      if cool_ratio >= 0.60: return "cool"
      return "mixed"

    def _derive_saturation(self, palette: List[Dict[str, Any]]) -> str:
      s_weighted_sum, w_sum = 0.0, 0.0

      for p in palette:
          w = float(p.get("proportion", 0.0))
          h, s, v = self._rgb_to_hsv(tuple(p["rgb"]))

          s_weighted_sum += s * w
          w_sum += w

      if w_sum <= 0: return "mixed"

      s_avg = s_weighted_sum / w_sum

      if s_avg >= 0.55: return "saturated"
      if s_avg <= 0.30: return "desaturated"
      return "mixed"

    def extra_info_inference(self, frames):
      total_palette = []

      for frame_bgr in frames:
          rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
          pil_img = Image.fromarray(rgb).resize(self.img_size, Image.BILINEAR)

          # 색상 추출 (최대 k개)
          colors = colorgram.extract(pil_img, self.cfg.palette_k)
          total_p = sum([c.proportion for c in colors]) or 1.0

          # 모든 프레임의 데이터를 하나의 리스트에 확장(extend)
          total_palette.extend([
              {
                  "rgb": [int(c.rgb.r), int(c.rgb.g), int(c.rgb.b)],
                  "proportion": float(c.proportion) / total_p
              } for c in colors
          ])

      # 통합 팔레트를 기반으로 최종 결과 도출
      warm_cool_result = self._derive_warm_cool(total_palette)
      saturation_result = self._derive_saturation(total_palette)

      return warm_cool_result, saturation_result

    def extract(self, video_path: str) -> Dict[str, Any]:
        frames = self._sample_frames_every(video_path, step_sec=self.cfg.sample_step_sec)

        fps_mode = int(round(1 / self.cfg.sample_step_sec)) if self.cfg.sample_step_sec > 0 else 1

        yolo_result = self.yolo_inference(
            video_path=video_path,
            frames=frames,
            fps_mode=fps_mode
        )
        warm_result, saturation_result = self.extra_info_inference(frames)
        result = {
            "top5" : yolo_result,
            "warm_cool": warm_result,
            "saturation": saturation_result
        }

        # save
        part = video_path.split('/')
        user_id = part[-2]
        video_name = part[-1]

        save_dir = os.path.join(os.getcwd(),"drive/MyDrive/study/extra_results", user_id)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f"{video_name}.json")

        with open(save_path, "w", encoding="utf-8") as f:
          json.dump(result, f, ensure_ascii=False, indent=2)

        return result

In [ ]:
from dataclasses import dataclass, asdict
from tqdm import tqdm

# =========================================================
# 4) Ray Workers
# =========================================================
@ray.remote(num_cpus=1)
class YOLOWorker:
    def __init__(self, config_dict: Dict[str, Any]):
        cfg = PipelineConfig(**config_dict)
        self.extractor = YOLOExtractor(cfg)

    def process(self, video_path: str) -> Dict[str, Any]:
        t0 = time.time()
        try:
            result = self.extractor.extract(video_path)
            return {
                "video_path": video_path,
                "task": "yolo",
                "status": "ok",
                "elapsed_sec": time.time() - t0,
                "result": result,
            }
        except Exception as e:
            return {
                "video_path": video_path,
                "task": "yolo",
                "status": "error",
                "elapsed_sec": time.time() - t0,
                "error": repr(e),
            }


@ray.remote(num_cpus=1, num_gpus=0.5)
class ShotVLWorker:
    def __init__(self, config_dict: Dict[str, Any]):
        cfg = PipelineConfig(**config_dict)
        self.extractor = ShotVLExtractor(cfg)

    def process(self, video_path: str) -> Dict[str, Any]:
        t0 = time.time()
        try:
            result = self.extractor.extract(video_path)
            return {
                "video_path": video_path,
                "task": "shotvl",
                "status": "ok",
                "elapsed_sec": time.time() - t0,
                "result": result,
            }
        except Exception as e:
            return {
                "video_path": video_path,
                "task": "shotvl",
                "status": "error",
                "elapsed_sec": time.time() - t0,
                "error": repr(e),
            }


# =========================================================
# 5) Helper: generic actor pool runner
# =========================================================
def run_actor_pool(
    worker_cls,
    num_workers: int,
    worker_config: Dict[str, Any],
    video_paths: List[str],
    label: str,
) -> List[Dict[str, Any]]:
    workers = [worker_cls.remote(worker_config) for _ in range(num_workers)]

    running: Dict[Any, Any] = {}
    video_iter = iter(video_paths)
    results: List[Dict[str, Any]] = []

    # 1. tqdm 바 설정 (전체 비디오 개수 기준)
    pbar = tqdm(total=len(video_paths), desc=f"[{label}] Processing")

    for worker in workers:
        try:
            vp = next(video_iter)
            ref = worker.process.remote(vp)
            running[ref] = worker
        except StopIteration:
            break

    while running:
        done_refs, _ = ray.wait(list(running.keys()), num_returns=4)
        done_ref = done_refs[0]
        worker = running.pop(done_ref)

        result = ray.get(done_ref)
        results.append(result)

        # 2. 작업 하나 완료될 때마다 pbar 갱신
        pbar.update(1)

        # 기존 print문은 tqdm 출력을 방해하지 않게 pbar.write()로 변경 권장
        if result["status"] == "ok":
            pbar.write(f"[{label}] DONE: {result['video_path']} ({result['elapsed_sec']:.2f}s)")
        else:
            pbar.write(f"[{label}] ERROR: {result['video_path']} -> {result.get('error')}")

        try:
            vp = next(video_iter)
            new_ref = worker.process.remote(vp)
            running[new_ref] = worker
        except StopIteration:
            pass

    pbar.close()
    return results

NameError: name 'ray' is not defined

In [ ]:
def run_yolo_session(video_paths, config_dict, num_workers=8):
    print(f"\n🚀 [Session 1] Starting YOLO Extraction (Workers: {num_workers})")
    yolo_results = run_actor_pool(
        worker_cls=YOLOWorker,
        num_workers=num_workers,
        worker_config=config_dict,
        video_paths=video_paths,
        label="YOLO"
    )
    return yolo_results

In [ ]:
def run_shotvl_session(video_paths, config_dict, num_workers=2):
    print(f"\n🚀 [Session 2] Starting ShotVL Extraction (Workers: {num_workers})")
    shotvl_results = run_actor_pool(
        worker_cls=ShotVLWorker,
        num_workers=num_workers,
        worker_config=config_dict,
        video_paths=video_paths,
        label="ShotVL"
    )
    return shotvl_results

In [ ]:
import torch


config = PipelineConfig()
config_dict = asdict(config)

# 모든 비디오 데이터 순회
video_paths = glob.glob(
    os.path.join(os.getcwd(), "drive/MyDrive/study/video_file/test/**/*.mp4"),
    recursive=True
)

# --- SESSION 1: YOLO ---
yolo_raw = run_yolo_session(video_paths, config_dict, num_workers=5)
# 중간 저장 (선택 사항)
yolo_map = {r["video_path"]: r for r in yolo_raw}

# --- SESSION 2: ShotVL ---
# shotvl_raw = run_shotvl_session(video_paths, config_dict, num_workers=2)
# shotvl_map = {r["video_path"]: r for r in shotvl_raw}

# --- FINAL MERGE ---
final_merged = []
for vp in video_paths:
    final_merged.append({
        "video_path": vp,
        "yolo": yolo_map.get(vp),
        #"shotvl": shotvl_map.get(vp)
    })

# 최종 저장
save_path = "final_yolo_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(final_merged, f, ensure_ascii=False, indent=2)

print(f"\n✅ All sessions completed. Results saved to {save_path}")


🚀 [Session 1] Starting YOLO Extraction (Workers: 5)


[YOLO] Processing:   0%|          | 0/2000 [00:00<?, ?it/s]

[YOLO] Processing:   0%|          | 1/2000 [01:25<47:36:08, 85.73s/it]

[YOLO] DONE: /content/drive/MyDrive/study/video_file/test/USER00002311/VIDEO00006856.mp4 (44.66s)


[YOLO] Processing:   0%|          | 2/2000 [01:26<19:56:55, 35.94s/it]

...

[YOLO] Processing:   0%|          | 4/2000 [02:00<12:37:09, 22.76s/it]

[YOLO] DONE: /content/drive/MyDrive/study/video_file/test/USER00002320/VIDEO00006352.mp4 (8.97s)


[YOLO] Processing:   0%|          | 5/2000 [02:11<10:19:46, 18.64s/it]

[YOLO] DONE: /content/drive/MyDrive/study/video_file/test/USER00002324/VIDEO00007268.mp4 (82.11s)


KeyboardInterrupt: 

In [ ]:
ls

drive/  final_ray_results.json  sample_data/  yolo26n.pt


In [ ]:
import torch
if __name__ == "__main__":
    # 0. 레이 초기화 및 모델 사전 점검
    if ray.is_initialized(): ray.shutdown()
    ray.init(num_cpus=12, num_gpus=1)

    config = PipelineConfig(
        shotvl_model_id="Vchitect/ShotVL-3B",
        shotvl_attn_implementation="sdpa",
        shotvl_dtype=torch.bfloat16,
        shotvl_max_new_tokens=64,
        shotvl_video_fps=1,
        shotvl_max_pixels=360 * 640,
        shotvl_device="cuda" if torch.cuda.is_available() else "cpu",
        yolo_model="yolo26n.pt",
        yolo_conf=0.65,
        yolo_device="cpu",
        palette_k=8,
        sample_step_sec=1.0,
    )
    config_dict = asdict(config)

    # 모든 비디오 데이터 순회
    video_paths = glob.glob(
        os.path.join(os.getcwd(), "drive/MyDrive/study/video_file/test/**/*.mp4"),
        recursive=True
    )

    # --- SESSION 1: YOLO ---
    # yolo_raw = run_yolo_session(video_paths, config_dict, num_workers=8)
    # # 중간 저장 (선택 사항)
    # yolo_map = {r["video_path"]: r for r in yolo_raw}

    # --- SESSION 2: ShotVL ---
    shotvl_raw = run_shotvl_session(video_paths, config_dict, num_workers=2)
    shotvl_map = {r["video_path"]: r for r in shotvl_raw}

    # --- FINAL MERGE ---
    final_merged = []
    for vp in video_paths:
        final_merged.append({
            "video_path": vp,
            #"yolo": yolo_map.get(vp),
            "shotvl": shotvl_map.get(vp)
        })

    # 최종 저장
    save_path = "final_ray_results.json"
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(final_merged, f, ensure_ascii=False, indent=2)

    print(f"\n✅ All sessions completed. Results saved to {save_path}")

2026-03-09 13:22:12,998	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 



🚀 [Session 2] Starting ShotVL Extraction (Workers: 2)



Loading weights: 100%|██████████| 824/824 [00:00<00:00, 1887.80it/s, Materializing param=model.visual.patch_embed.proj.weight]
(ShotVLWorker pid=29179) qwen-vl-utils using torchcodec to read video.
Loading weights:  95%|█████████▌| 786/824 [00:00<00:00, 1657.84it/s, Materializing param=model.visual.blocks.29.attn.qkv.weight] [repeated 3x across cluster]


[ShotVL] Processing:   0%|          | 1/2000 [00:32<17:52:42, 32.20s/it]

[ShotVL] DONE: /content/drive/MyDrive/study/video_file/test/USER00002311/VIDEO00006856.mp4 (15.96s)




[ShotVL] Processing:   0%|          | 2/2000 [00:35<8:35:28, 15.48s/it]

...

[ShotVL] DONE: /content/drive/MyDrive/study/video_file/test/USER00002309/VIDEO00006578.mp4 (6.76s)




[ShotVL] Processing: 100%|█████████▉| 1999/2000 [2:46:34<00:06,  6.00s/it]

[ShotVL] DONE: /content/drive/MyDrive/study/video_file/test/USER00002308/VIDEO00005468.mp4 (17.19s)




[ShotVL] Processing: 100%|██████████| 2000/2000 [2:46:35<00:00,  5.00s/it]

[ShotVL] DONE: /content/drive/MyDrive/study/video_file/test/USER00002310/VIDEO00005567.mp4 (5.49s)

✅ All sessions completed. Results saved to final_ray_results.json
